# 02 · Baseline logistic regression, one 70 / 15 / 15 split

The simplest honest setup: Hospital A's patients are split once into
**70% training**, **15% validation** and **15% test**. The model is trained on the training patients and scored on the validation patients. The test patients are set aside and not scored here.

Model: logistic regression with `class_weight="balanced"` on all 40 features (minus the ones that are at least 99% missing). Preprocessing is the same as in every other notebook and lives in `sepsis.py`:
forward fill inside each patient → training medians for what is still missing (unknown ICU unit = -1) → standardization with the training mean and std.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

import sepsis as s

data_A = s.load_hospital("A")
print(f"Hospital A: {data_A['PatientID'].nunique():,} patients, {len(data_A):,} hourly rows")

Hospital A: 20,336 patients, 790,215 hourly rows


## 1. Split the patients

In [2]:
# Step 1: set the 15% TEST patients aside first, so nothing below can touch them
trainval_ids, test_ids = s.split_patients(data_A, test_size=0.15)

# Step 2: split the remaining 85% again. 0.15 / 0.85 of it = 15% of ALL patients
train_ids, val_ids = s.split_patients(data_A, test_size=0.15 / 0.85, ids=trainval_ids)

assert set(train_ids).isdisjoint(val_ids) and set(trainval_ids).isdisjoint(test_ids)

status = s.patient_status(data_A)
for name, ids in [("Training", train_ids), ("Validation", val_ids), ("Test (not scored here)", test_ids)]:
    rows = data_A["PatientID"].isin(ids).sum()
    print(f"{name}: {len(ids):,} patients ({int(status.loc[ids].sum())} septic), {rows:,} hourly rows")

Training: 14,234 patients (1253 septic), 553,696 hourly rows
Validation: 3,051 patients (268 septic), 117,616 hourly rows
Test (not scored here): 3,051 patients (269 septic), 118,903 hourly rows


## 2. Train and predict

In [3]:
# Only training + validation patients are prepared; the test patients are left out completely.
# (No folds here, and no missingness features: this is the plain baseline.)
dev = s.prepare_dev(data_A, np.concatenate([train_ids, val_ids]), missingness=False)

train = np.isin(dev["patients"], train_ids)
val = ~train

y_prob = s.fit_predict(dev, train, val, s.ALL_FEATURES)        # P(sepsis) for every validation hour
y_val = dev["labels"][val]

trained on 36 input features


## 3. Results on the validation set

In [4]:
utility = s.make_utility_scorer(dev["patients"][val], y_val, dev["hours"][val])
y_pred = (y_prob >= 0.5).astype(int)

# AUROC and AUPRC use the probabilities directly: no cutoff needed
print("AUROC:", round(roc_auc_score(y_val, y_prob), 4))
print("AUPRC:", round(average_precision_score(y_val, y_prob), 4))

print("\nAt cutoff 0.5:")
for name, value in s.threshold_metrics(y_val, y_pred).items():
    print(f"{name}: {value:.4f}")
print("Clinical utility:", round(utility(y_pred), 4))

print("\nActual positive hours:", int(y_val.sum()))
print("Predicted positive hours:", int(y_pred.sum()))
print("\nConfusion matrix [[TN FP] [FN TP]]:")
print(confusion_matrix(y_val, y_pred))

AUROC: 0.7433
AUPRC: 0.0861

At cutoff 0.5:
F1: 0.0941
Precision: 0.0509
Recall: 0.6143
FPR: 0.2551
Clinical utility: 0.2898

Actual positive hours: 2564
Predicted positive hours: 30922

Confusion matrix [[TN FP] [FN TP]]:
[[85705 29347]
 [  989  1575]]


In [5]:
# The cutoff decides how many alarms are raised. Lower = more septic hours caught, but many more false alarms.
rows = []
for cutoff in [0.90, 0.80, 0.70, 0.65, 0.60, 0.55, 0.50, 0.40, 0.30, 0.20, 0.10]:
    pred = (y_prob >= cutoff).astype(int)
    rows.append({"Cutoff": cutoff, **s.threshold_metrics(y_val, pred),
                 "Clinical utility": utility(pred),
                 "Positive predictions": int(pred.sum()),
                 "False positive hours": int(((pred == 1) & (y_val == 0)).sum())})

pd.DataFrame(rows).set_index("Cutoff").round(4)

,F1,Precision,Recall,FPR,Clinical utility,Positive predictions,False positive hours
Cutoff,,,,,,,
0.90,0.1152,0.1900,0.0827,0.0079,0.0762,1116,904
0.80,0.1711,0.1495,0.2001,0.0254,0.1706,3431,2918
0.70,0.1575,0.1042,0.3222,0.0617,0.2483,7926,7100
0.65,0.1452,0.0888,0.3986,0.0912,0.2863,11510,10488
0.60,0.1269,0.0733,0.4707,0.1325,0.3069,16456,15249
0.55,0.1095,0.0609,0.5410,0.1859,0.3077,22773,21386
0.50,0.0941,0.0509,0.6143,0.2551,0.2898,30922,29347
0.40,0.0718,0.0376,0.7785,0.4436,0.2064,53031,51035
0.30,0.0545,0.0281,0.9021,0.6958,-0.0006,82365,80052


## Conclusion

The baseline logistic regression model was evaluated on the validation set only (15% of Hospital A's patients). The test set (15% of Hospital A) and Hospital B are kept untouched until the final evaluation. The outputs above report AUROC and AUPRC, and, at each cutoff, F1, precision, recall, false positive rate (FPR) and the clinical utility score.